## 3. HMM Event Sequence Classification — Teradata Vantage

**Data:** `banking_v1_train` / `banking_v1_test`

| Step | What |
|------|------|
| 1 | Connect to Vantage |
| 2 | Create HMM tables |
| 3 | Install 8 stored procedures |
| 4 | Binary HMM (ApplyMortgage vs NoApplication) — 3/4/5 states |
| 5 | Multiclass HMM (all Apply\* products) — 3 states |
| 6 | Full metrics & visualization |
| 7 | Model interpretation (emissions, transitions) |

All SQL, SP definitions, training/scoring workflows, and plotting live in **`hmm_support.py`**.

## 1. Setup & Connect

In [1]:
from hmm_support import *

In [2]:
ctx = connect_vantage("vantage24.td.teradata.com", "kg255057")

Enter LDAP password for kg255057@vantage24.td.teradata.com:  ········


Connecting to vantage24.td.teradata.com as kg255057 (LDAP)...
Connected. Database=KG255057, User=KG255057, Session=4962573


In [3]:
verify_source_tables("banking_v1_train", "banking_v1_test")

  banking_v1_train: 1,499,288 rows
  banking_v1_test: 599,733 rows


## 2. Create Tables & Install Stored Procedures

In [4]:
create_hmm_tables(drop_existing=True)

Creating HMM tables...
  Created hmm_params_pi
  Created hmm_params_trans
  Created hmm_params_emit
  Created hmm_sequences
  Created hmm_alpha
  Created hmm_beta
  Created hmm_gamma
  Created hmm_xi
  Created hmm_train_log
  Created hmm_scores


In [5]:
install_all_stored_procedures()

Installing stored procedures...
  Installed SP_HMM_EXTRACT_SEQUENCES
  Installed SP_HMM_INIT
  Installed SP_HMM_FORWARD
  Installed SP_HMM_BACKWARD
  Installed SP_HMM_ESTEP
  Installed SP_HMM_MSTEP
  Installed SP_HMM_TRAIN
  Installed SP_HMM_SCORE_SESSIONS


## 3. Binary Classification: ApplyMortgage vs NoApplication

Train a positive HMM on sessions that end in `ApplyMortgage` and a negative
HMM on sessions with no Apply event. Score the test set with both, compute
log-likelihood ratio → sigmoid → probability. Repeat for 3, 4, and 5 states.

In [6]:
TARGET = 'ApplyMortgage'
TRAIN_TABLE = 'banking_v1_train'
TEST_TABLE = 'banking_v1_test'

binary_results = {}

for n_states in [3, 4, 5]:
    print(f"\n{'='*60}")
    print(f"  {n_states} HIDDEN STATES")
    print(f"{'='*60}")

    # Train
    pos_id, neg_id = train_binary_hmm(TARGET, TRAIN_TABLE, n_states=n_states)

    # Score
    scores = score_binary_hmm(pos_id, neg_id, TEST_TABLE)

    # Evaluate
    metrics = evaluate_binary(scores, TEST_TABLE, TARGET)

    binary_results[n_states] = metrics


  3 HIDDEN STATES

Training binary HMM for 'ApplyMortgage' (3 states)...
  Positive class... done (22.8s)
  Negative class... done (1116.0s)
  bin_ApplyMortgage_pos_3s: 3 iters, final ll=-21.2119
  bin_ApplyMortgage_neg_3s: 3 iters, final ll=-28.1308
  Score positive... done (38.5s)
  Score negative... done (43.4s)

  Binary evaluation (65488 sessions, 1331 positive):
    Th=0.3: Acc=0.855 P=0.096 R=0.726 F1=0.169
    Th=0.5: Acc=0.962 P=0.306 R=0.687 F1=0.423
    Th=0.7: Acc=0.985 P=0.614 R=0.678 F1=0.645
    AUC-ROC: 0.8464

  4 HIDDEN STATES

Training binary HMM for 'ApplyMortgage' (4 states)...
  Positive class... done (88.9s)
  Negative class... done (771.4s)
  bin_ApplyMortgage_pos_4s: 3 iters, final ll=-21.2119
  bin_ApplyMortgage_neg_4s: 3 iters, final ll=-28.1308
  Score positive... done (52.6s)
  Score negative... done (65.6s)

  Binary evaluation (65488 sessions, 1331 positive):
    Th=0.3: Acc=0.855 P=0.096 R=0.726 F1=0.169
    Th=0.5: Acc=0.962 P=0.306 R=0.687 F1=0.423
  

### Binary Results Summary

In [7]:
rows = []
for ns, r in binary_results.items():
    m05 = r['thresholds'][0.5]
    rows.append({
        'States': ns, 'AUC': r['auc'],
        'Accuracy': m05['accuracy'], 'Precision': m05['precision'],
        'Recall': m05['recall'], 'F1': m05['f1']
    })
print(pd.DataFrame(rows).to_string(index=False))

 States      AUC  Accuracy  Precision   Recall       F1
      3 0.846371  0.961932   0.305816 0.687453 0.423317
      4 0.846371  0.961932   0.305816 0.687453 0.423317
      5 0.846371  0.961932   0.305816 0.687453 0.423317


In [8]:
plot_binary_results(binary_results, TARGET)

  Saved hmm_binary_results.png


## 4. Multiclass Classification: All Apply\* Products

Train one HMM per Apply\* class plus NoApplication. Classify each test session
by assigning it to whichever class HMM gives the highest log-likelihood.

In [9]:
mc_model_ids = train_multiclass_hmm(TRAIN_TABLE, n_states=3)


Training multiclass HMMs (3 states) for 7 Apply classes + NoApplication...
  ApplyAutoLoan... done (150.8s)
  ApplyCheckingAccount... done (138.3s)
  ApplyCreditCard... done (205.0s)
  ApplyMortgage... done (148.6s)
  ApplyPersonalLoan... done (151.3s)
  ApplySavingsAccount... done (179.5s)
  ApplyTeenChecking... done (119.8s)
  NoApplication... done (750.8s)
  ApplyAutoLoan: 3 iters, ll=-20.3195
  ApplyCheckingAccount: 3 iters, ll=-19.1651
  ApplyCreditCard: 3 iters, ll=-21.0833
  ApplyMortgage: 3 iters, ll=-21.2119
  ApplyPersonalLoan: 3 iters, ll=-20.7724
  ApplySavingsAccount: 3 iters, ll=-21.1480
  ApplyTeenChecking: 3 iters, ll=-18.1092
  NoApplication: 3 iters, ll=-28.1308


In [10]:
mc_scores, mc_class_names = score_multiclass_hmm(mc_model_ids, TEST_TABLE)


Scoring test set against 8 class models...
  ApplyAutoLoan... done (98.7s)
  ApplyCheckingAccount... done (74.1s)
  ApplyCreditCard... done (66.2s)
  ApplyMortgage... done (67.0s)
  ApplyPersonalLoan... done (76.6s)
  ApplySavingsAccount... done (77.3s)
  ApplyTeenChecking... done (71.7s)
  NoApplication... done (88.0s)


In [11]:
mc_results = evaluate_multiclass(mc_scores, TEST_TABLE)


  Multiclass evaluation (81081 sessions):
  Overall accuracy: 0.8611
                      precision    recall  f1-score   support

       ApplyAutoLoan       0.65      0.69      0.67      1358
ApplyCheckingAccount       0.40      0.69      0.50      1252
     ApplyCreditCard       0.60      0.68      0.64      3796
       ApplyMortgage       0.49      0.68      0.57      1331
   ApplyPersonalLoan       0.43      0.68      0.52      1244
 ApplySavingsAccount       0.70      0.69      0.70      3165
   ApplyTeenChecking       0.34      0.70      0.46       642
       NoApplication       0.94      0.89      0.92     68293

            accuracy                           0.86     81081
           macro avg       0.57      0.71      0.62     81081
        weighted avg       0.88      0.86      0.87     81081



In [12]:
plot_multiclass_results(mc_results)

  Saved hmm_multiclass_results.png


## 5. Training Convergence

In [13]:
# Combine all model_ids for convergence plot
all_model_ids = {}
for cls, mid in mc_model_ids.items():
    all_model_ids[cls] = mid
# Add binary models from best state count
best_ns = max(binary_results, key=lambda k: binary_results[k]['auc'])
all_model_ids[f'{TARGET}_pos'] = f'bin_{TARGET}_pos_{best_ns}s'
all_model_ids[f'{TARGET}_neg'] = f'bin_{TARGET}_neg_{best_ns}s'

plot_training_convergence(all_model_ids)

  Saved hmm_convergence.png


## 6. Combined Results Plot

In [14]:
plot_all_results(binary_results, mc_results, TARGET)

  Saved hmm_teradata_all_results.png


## 7. Model Interpretation

What do the hidden states *mean*? Examine the top emission events per state
and the transition dynamics to understand customer journey phases.

In [15]:
for cls in ['ApplyMortgage', 'ApplyCreditCard', 'NoApplication']:
    mid = mc_model_ids.get(cls)
    if mid:
        print(f"\n{'='*50}")
        print(f"  {cls}")
        print(f"{'='*50}")
        print_model_interpretation(mid)


  ApplyMortgage

  Top emissions for mc_ApplyMortgage_3s:
    State 1: MortgageRateInquiry(0.066), PreQualificationInquiry(0.061), DiscussHomeFinancing(0.045), ReviewMortgageDocuments(0.043), InquireMortgageOptions(0.043), ViewRewards(0.032), ViewSavings(0.032), ViewStatements(0.031)
    State 2: MortgageRateInquiry(0.066), PreQualificationInquiry(0.061), DiscussHomeFinancing(0.045), ReviewMortgageDocuments(0.043), InquireMortgageOptions(0.043), ViewRewards(0.032), ViewSavings(0.032), ViewStatements(0.031)
    State 3: MortgageRateInquiry(0.066), PreQualificationInquiry(0.061), DiscussHomeFinancing(0.045), ReviewMortgageDocuments(0.043), InquireMortgageOptions(0.043), ViewRewards(0.032), ViewSavings(0.032), ViewStatements(0.031)
  Transitions:
    1 -> [0.600 0.200 0.200]
    2 -> [0.200 0.600 0.200]
    3 -> [0.200 0.200 0.600]

  ApplyCreditCard

  Top emissions for mc_ApplyCreditCard_3s:
    State 1: CompareCreditCards(0.053), CheckCreditScore(0.052), CreditLimitInquiry(0.048), Cre

## 8. Cleanup (Optional)

Uncomment to remove all HMM artifacts from Vantage.

In [16]:
# drop_hmm_tables()
# drop_all_stored_procedures()
# disconnect()